In [5]:
import numpy as np 
import pandas as pd 
from time import perf_counter_ns

In [2]:
A = np.array(np.random.randint(0, 10, (3, 3)), dtype = np.float64)
Q, R = np.linalg.qr(A)
print("Q: ")
print(Q)
print("R: ")
print(R)

qqt = np.dot(Q, Q.T)
qtq = np.dot(Q.T, Q)
print("Q Q**T: ")
print(np.round(qqt))
print("Q**T Q:")
print(np.round(qtq))

Q: 
[[ 0.         -0.49402355 -0.86944852]
 [-0.40613847 -0.7945121   0.45144442]
 [-0.91381155  0.35311649 -0.20064197]]
R: 
[[ -9.8488578   -7.81816547 -11.0672732 ]
 [  0.          -6.07258501  -3.3715834 ]
 [  0.           0.          -0.38456377]]
Q Q**T: 
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Q**T Q:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [9]:
def gram_schmidt(A):
    m, n = A.shape
    Q = np.zeros((m, n))
    Q[:, 0] = A[:, 0] / np.linalg.norm(A[:, 0], 2)
    for i in range(1, n):
        Q[:, i] = A[:, i]
        for j in range(0, i):
            inner = np.dot(Q[:, j].T, Q[:, i])
            Q[:, i] = Q[:, i] - np.dot(inner, Q[:, j])
        Q[:, i] = Q[:, i] / np.linalg.norm(Q[:, i], 2)
    return Q

def qr_gs(A):
    m, n = A.shape
    Q = gram_schmidt(A)
    R = np.zeros((m, n))
    for i in range(0, n):
        R[i, i] = np.dot(Q[:, i], A[:, i])
        for j in range(0, i):
            R[j, i] = np.dot(Q[:, j], A[:, i])
    return Q, R

start = perf_counter_ns()
Q, R = qr_gs(A)
end = perf_counter_ns()
print("Q: ")
print(Q)
print("R: ")
print(R)
print(f"Runtime: {end - start} ns") 

Q: 
[[ 0.          0.49402355  0.86944852]
 [ 0.40613847  0.7945121  -0.45144442]
 [ 0.91381155 -0.35311649  0.20064197]]
R: 
[[ 9.8488578   7.81816547 11.0672732 ]
 [ 0.          6.07258501  3.3715834 ]
 [ 0.          0.          0.38456377]]
Runtime: 288459 ns


In [11]:
def qr_householder(A):
    m, n = A.shape
    R = A.copy()
    Q = np.identity(m)
    for i in range(0, n - 1): 
        alpha = np.linalg.norm(R[i:, i], 2)
        avec = np.zeros_like(R[i:, [i]])
        avec[0] = alpha
        u = R[i:, [i]] - avec
        v = u . np.linalg.norm(u, 2)
        Qn  = np.identity(m - i) - (2 * np.dot(v, v.T))
        Qn = np.block([[np.eye(i), np.zeros((i, m - i))],
                      [np.zeros((m - i, i)), Qn]])
        R = np.dot(Qn, R)
        Q = np.dot(Q, Qn.T)

    return Q, R

start = perf_counter_ns()
Q, R = qr_gs(A)
end = perf_counter_ns()
print("Q: ")
print(Q)
print("R: ")
print(R)
print(f"Runtime: {end - start} ns") 

Q: 
[[ 0.          0.49402355  0.86944852]
 [ 0.40613847  0.7945121  -0.45144442]
 [ 0.91381155 -0.35311649  0.20064197]]
R: 
[[ 9.8488578   7.81816547 11.0672732 ]
 [ 0.          6.07258501  3.3715834 ]
 [ 0.          0.          0.38456377]]
Runtime: 2983041 ns


In [12]:
def givensrotation(a, b):
    hypot = np.sqrt(a**2 + b**2)
    cos = a / hypot
    sin = -b / hypot
    return cos, sin


def qr_givens(A):
    m, n = A.shape
    R = A.copy()
    Q = np.identity(m)
    for i in range(0, n - 1):
        for j in range(i + 1, m):
            cos, sin = givensrotation(R[i, i], R[j, i])
            R[i], R[j] = (R[i] * cos) + (R[j] * (-sin)), (R[i] * sin) + (R[j] * cos)
            Q[:, i], Q[:, j] = (Q[:, i] * cos) + (Q[:, j] * (-sin)), (Q[:, i] * sin) + (Q[:, j] * cos)
    return Q, R

start = perf_counter_ns()
Q, R = qr_givens(A)
end = perf_counter_ns()
print("Q:")
print(Q)
print("R:")
print(np.round(R, 8))
print(f"Runtime: {end - start} ns")

qqt = np.dot(Q, Q.T)
qtq = np.dot(Q.T, Q)
print("Q Q**T:")
print(np.round(qqt))
print("Q**T Q:")
print(np.round(qtq))

Q:
[[ 0.          0.49402355 -0.86944852]
 [ 0.40613847  0.7945121   0.45144442]
 [ 0.91381155 -0.35311649 -0.20064197]]
R:
[[ 9.8488578   7.81816547 11.0672732 ]
 [-0.          6.07258501  3.3715834 ]
 [ 0.          0.         -0.38456377]]
Runtime: 7098125 ns
Q Q**T:
[[ 1.  0.  0.]
 [ 0.  1. -0.]
 [ 0. -0.  1.]]
Q**T Q:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [13]:
A = np.array([ [1, -1]]).T
Q, R = np.linalg.qr(A, 'complete')
Q * np.sqrt(2)



array([[-1.,  1.],
       [ 1.,  1.]])